Load Dataset

In [1]:
import pandas as pd

df = pd.read_csv("../data/usgs_2025.csv")

Cek Ukuran, Profil, Statistik Dataset

In [2]:
df.shape 
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139939 entries, 0 to 139938
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   id         139939 non-null  object 
 1   magnitude  139930 non-null  float64
 2   place      139939 non-null  object 
 3   time       139939 non-null  int64  
 4   updated    139939 non-null  int64  
 5   longitude  139939 non-null  float64
 6   latitude   139939 non-null  float64
 7   depth      139939 non-null  float64
dtypes: float64(4), int64(2), object(2)
memory usage: 8.5+ MB


,magnitude,time,updated,longitude,latitude,depth
count,139930.000000,1.399390e+05,1.399390e+05,139939.000000,139939.000000,139939.000000
mean,1.699608,1.751357e+12,1.753584e+12,-101.472099,40.070126,22.450245
std,1.386042,9.036023e+09,9.781731e+09,83.132062,19.236679,54.611416
min,-5.000000,1.735690e+12,1.735692e+12,-179.995100,-73.220400,-3.500000
25%,0.780000,1.743459e+12,1.745349e+12,-150.431600,33.788583,3.390000
50%,1.480000,1.751976e+12,1.753394e+12,-121.764500,38.825832,7.850000
75%,2.200000,1.758582e+12,1.760938e+12,-104.433000,54.353300,15.090000
max,8.800000,1.767139e+12,1.781332e+12,179.999400,87.081500,669.556000


Cek Nilai Hilang Dataset

In [3]:
df.isnull().sum()
df = df.dropna()

Konversi Timetemps

In [4]:
df["time"] = pd.to_datetime(
    df["time"],
    unit="ms"
)

df.head()

,id,magnitude,place,time,updated,longitude,latitude,depth
0,us7000pc44,4.50,"74 km WNW of Lata, Solomon Islands",2025-01-31 23:57:39.481,1744750533040,165.158400,-10.483900,49.061
1,av93479546,-0.03,"63 km WNW of Tyonek, Alaska",2025-01-31 23:54:22.350,1738398201080,-152.243667,61.259500,3.060
2,ak0251fnn3q1,1.40,"11 km SW of Takotna, Alaska",2025-01-31 23:48:40.578,1740462233108,-156.245800,62.921600,8.900
3,ak0251fnn3pu,3.50,"22 km S of King Salmon, Alaska",2025-01-31 23:48:32.250,1744750526040,-156.619100,58.490300,199.700
4,uw62065217,0.53,"17 km NNE of Ashford, Washington",2025-01-31 23:43:51.070,1738611706810,-121.956500,46.907667,4.420


Feature Engineering (Ekstrak Tanggal, Waktu, Lokasi) serta Drop Kolom

In [ ]:
# Menghapus Beberapa Kolom yang tidak dipakai dalam Analsis
if 'updated' in df.columns:
    df = df.drop(columns=['updated'])

# 2. Mengekstrak Tanggal (tahun, bulan, tanggal)
df['tahun'] = df['time'].dt.year
df['bulan'] = df['time'].dt.month
df['tanggal'] = df['time'].dt.day

# 3. Mengekstrak Waktu (jam, menit)
df['jam'] = df['time'].dt.hour
df['menit'] = df['time'].dt.minute

# 4. Mengekstrak fitur dari kolom 'place'
df['jarak_km'] = df['place'].str.extract(r'(\d+(?:\.\d+)?)\s*km').astype(float)
df['arah'] = df['place'].str.extract(r'km\s+([a-zA-Z]+)\s+of')
df['kota_terdekat'] = df['place'].str.extract(r'of\s+([^,]+)')
df['wilayah'] = df['place'].str.split(',').str[-1].str.strip()

# 5. Menghapus kolom 'time' dan 'place' karena sudah dipecah
df = df.drop(columns=['time', 'place'])

df.head()


,id,magnitude,longitude,latitude,depth,tahun,bulan,tanggal,jam,menit,jarak_km,arah,kota_terdekat,wilayah
0,us7000pc44,4.50,165.158400,-10.483900,49.061,2025,1,31,23,57,74.0,WNW,Lata,Solomon Islands
1,av93479546,-0.03,-152.243667,61.259500,3.060,2025,1,31,23,54,63.0,WNW,Tyonek,Alaska
2,ak0251fnn3q1,1.40,-156.245800,62.921600,8.900,2025,1,31,23,48,11.0,SW,Takotna,Alaska
3,ak0251fnn3pu,3.50,-156.619100,58.490300,199.700,2025,1,31,23,48,22.0,S,King Salmon,Alaska
4,uw62065217,0.53,-121.956500,46.907667,4.420,2025,1,31,23,43,17.0,NNE,Ashford,Washington


Klasifikasi Magnitude

In [6]:
#Membuat Kategori Gempa
def kategori_mag(mag):

    if mag < 3:
        return "Kecil"

    elif mag < 5:
        return "Sedang"

    elif mag < 7:
        return "Besar"

    else:
        return "Sangat Besar"
    
#Menambahkan Kolom Kategori
df["kategori"] = df["magnitude"].apply(
    kategori_mag
)
df["kategori"].value_counts()


kategori
Kecil           117157
Sedang           20647
Besar             2110
Sangat Besar        16
Name: count, dtype: int64

Menyimpan Dataset Final

In [7]:
df.to_csv(
    "../data/usgs_2025_final.csv",
    index=False
)